# LeetCode 3449 — Maximize the Minimum Game Score bằng Python

- Link: https://leetcode.com/problems/maximize-the-minimum-game-score/
- Độ khó: **Hard**
- Chủ đề: **Tìm kiếm nhị phân đáp án** + **tham lam** đi "qua lại" giữa `i` và `i + 1` để mỗi ô đủ số lần ghé

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def maxScore(self, points: List[int], m: int) -> int:
```

## 1. Mục tiêu học

1. Đổi bài "tối đa hóa giá trị nhỏ nhất" thành câu hỏi có/không: "có thể làm mọi `gameScore[i] ≥ x` với ≤ `m` bước?" ⇒ nhị phân `x`.
2. Đổi điều kiện điểm thành **số lần ghé**: ô `i` cần `need[i] = ⌈x / points[i]⌉` lần ghé.
3. Chiến lược tham lam trái → phải: đi tới `i`, rồi **nảy qua lại** `i ↔ i + 1` cho đủ lần ghé ở `i`; mỗi lần nảy cũng cho `i + 1` một lần ghé "miễn phí".
4. Xử lý ô cuối: nếu đã đủ nhờ các lần nảy từ `i − 1` thì **không cần bước vào** nữa.
5. Chọn cận trên nhị phân đúng (`min(points) · m`) và kiểm chứng bằng vét cạn mọi dãy bước.

## 2. Đề bài & ví dụ

Mảng `points` độ dài `n`, mảng `gameScore` ban đầu toàn 0. Bắt đầu ở chỉ số `−1`. Tối đa `m` bước; mỗi bước tăng hoặc giảm chỉ số 1 (sau bước đầu phải luôn trong `[0, n − 1]`) và **cộng `points[i]` vào `gameScore[i]`** tại ô vừa tới. Trả về giá trị **lớn nhất có thể của `min(gameScore)`**.

**Ví dụ 1:** `points = [2, 4], m = 3` → `4` (0 → 1 → 0: điểm `[4, 4]`).

**Ví dụ 2:** `points = [1, 2, 3], m = 5` → `2`.

**Ràng buộc:** `2 ≤ n ≤ 5·10^4`, `1 ≤ points[i] ≤ 10^6`, `1 ≤ m ≤ 10^9`.

In [1]:
from typing import List

examples = [([2, 4], 3, 4), ([1, 2, 3], 5, 2)]
for p, m, exp in examples:
    print(p, m, '-> kỳ vọng', exp)

[2, 4] 3 -> kỳ vọng 4
[1, 2, 3] 5 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. Mô phỏng một dãy bước

In [2]:
def simulate(points, moves):
    # moves: chuỗi 'R' (tăng) / 'L' (giảm); trả về gameScore hoặc None nếu ra ngoài
    n, i = len(points), -1
    score = [0] * n
    for c in moves:
        i += 1 if c == 'R' else -1
        if not 0 <= i < n:
            return None
        score[i] += points[i]
    return score

print(simulate([2, 4], 'RRL'), simulate([1, 2, 3], 'RRLRR'), simulate([1, 2], 'L'))

[4, 4] [2, 4, 3] None


### 3.2. Từ điểm sang số lần ghé

`gameScore[i] = points[i] · (số lần ghé ô i)` ⇒ `gameScore[i] ≥ x` ⇔ số lần ghé ≥ `⌈x / points[i]⌉`. Trong Python: `(x + p − 1) // p`.

### 3.3. Tìm kiếm nhị phân trên đáp án

`ok(x)` đơn điệu: làm được `x` thì làm được mọi `x' < x`. Cận dưới `0` (luôn được), cận trên `min(points) · m` (ô có điểm nhỏ nhất được ghé tối đa `m` lần).

In [3]:
def max_true(lo, hi, ok):
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid):
            lo = mid
        else:
            hi = mid - 1
    return lo

print(max_true(0, 100, lambda x: x * x <= 50))

7


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2 với `x = 2`: `need = [2, 1, 1]`.

- Ô 0: bước vào (1 bước, 1 lần ghé). Cần thêm 1 ⇒ nảy `0 → 1 → 0` (2 bước) — ô 1 được ghé "ké" 1 lần. Tổng 3 bước.
- Ô 1: cần `1 − 1 = 0` nữa, nhưng vẫn phải **đi qua** để tới ô 2 ⇒ bước vào ô 1 (1 bước). Tổng 4.
- Ô 2: cần 1 ⇒ bước vào (1 bước). Tổng 5 ≤ 5 ✓.

Với `x = 3`: `need = [3, 2, 1]`: ô 0: `1 + 2·2 = 5` bước (ô 1 được ké 2); ô 1: phải đi qua: 1 ⇒ 6 > 5 ✗. ⇒ đáp án 2.

In [4]:
print([(2 + p - 1) // p for p in [1, 2, 3]], [(3 + p - 1) // p for p in [1, 2, 3]])

[2, 1, 1] [3, 2, 1]


## 5. Cách 1 — Brute Force: thử mọi dãy bước

In [5]:
def max_score_brute(points, m):
    n = len(points)
    best = 0
    def dfs(pos, used, score):
        nonlocal best
        best = max(best, min(score))
        if used == m:
            return
        for q in (pos - 1, pos + 1):
            if 0 <= q < n:
                s = list(score); s[q] += points[q]
                dfs(q, used + 1, s)
    dfs(-1, 0, [0] * n)
    return best

for p, m, exp in examples:
    print(max_score_brute(p, m), '| kỳ vọng', exp)

4 | kỳ vọng 4
2 | kỳ vọng 2


**Độ phức tạp:** `O(2^m · n)` — chỉ dùng cho `m ≤ 12`.

## 6. Ý tưởng thuật toán / bước đột phá

**Hàm kiểm tra `ok(x)`** (tham lam trái → phải), biến `prev` = số lần ô `i` đã được ghé "ké" nhờ nảy từ ô `i − 1`:

```
moves = 0, prev = 0
for i in 0..n-1:
    k = ⌈x / p[i]⌉ − prev          # số lần ghé còn thiếu ở ô i
    if i == n−1 and k ≤ 0: break    # ô cuối đã đủ, không cần bước vào
    k = max(k, 1)                   # phải bước vào ô i ít nhất 1 lần để đi tiếp
    moves += 2k − 1                 # 1 bước vào + (k−1) lần nảy, mỗi lần 2 bước
    prev = k − 1                    # mỗi lần nảy ghé ô i+1 một lần
    if moves > m: return False
return True
```

Ô cuối nảy với ô `n − 2` (thay vì `n`) — cùng chi phí 2 bước mỗi lần ghé thêm.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Hàm kiểm tra tham lam + dò `x` tuyến tính

Dò `x = 1, 2, …` cho tới khi `ok(x)` sai. Đúng nhưng đáp án có thể tới `~10^{15}` ⇒ chỉ dùng khi đáp án nhỏ (minh họa tính đơn điệu).

In [6]:
def ok_greedy(points, m, x):
    n = len(points)
    moves, prev = 0, 0
    for i in range(n):
        k = (x + points[i] - 1) // points[i] - prev
        if i == n - 1 and k <= 0:
            break
        k = max(k, 1)
        moves += 2 * k - 1
        prev = k - 1
        if moves > m:
            return False
    return True

def max_score_linear(points, m, limit=10 ** 6):
    x = 0
    while x < limit and ok_greedy(points, m, x + 1):
        x += 1
    return x

for p, m, exp in examples:
    print(max_score_linear(p, m), '| kỳ vọng', exp)

4 | kỳ vọng 4
2 | kỳ vọng 2


### 7.2. Cách 3 — Nhị phân + tham lam (lời giải chính)

In [7]:
def max_score_bs(points, m):
    return max_true(0, min(points) * m, lambda x: ok_greedy(points, m, x))

for p, m, exp in examples:
    print(max_score_bs(p, m), '| kỳ vọng', exp)

4 | kỳ vọng 4
2 | kỳ vọng 2


### 7.3. Cách 4 — Dựng đường đi cụ thể để kiểm chứng

Với `x` cho trước, dựng **chuỗi bước** đúng như chiến lược tham lam (`'R'`, `'L'`), rồi mô phỏng để chắc chắn mọi ô đạt `≥ x` và số bước `≤ m`. Dùng cho `m` nhỏ/vừa — vừa kiểm chứng hàm kiểm tra, vừa cho ra lời giải cụ thể.

In [8]:
def build_moves(points, x):
    n = len(points)
    out, prev = [], 0
    for i in range(n):
        k = (x + points[i] - 1) // points[i] - prev
        if i == n - 1 and k <= 0:
            break
        k = max(k, 1)
        out.append('R')                                  # bước vào ô i
        if i < n - 1:
            out.extend(['R', 'L'] * (k - 1))             # nảy i -> i+1 -> i
        else:
            out.extend(['L', 'R'] * (k - 1))             # ô cuối nảy với ô n-2
        prev = k - 1
    return ''.join(out)

def max_score_constructive(points, m):
    def ok(x):
        mv = build_moves(points, x)
        if len(mv) > m:
            return False
        sc = simulate(points, mv)
        return sc is not None and min(sc) >= x
    return max_true(0, min(points) * m, ok)

for p, m, exp in examples:
    x = max_score_constructive(p, m)
    print(x, '| kỳ vọng', exp, '| đường đi:', build_moves(p, x), '->', simulate(p, build_moves(p, x)))

4 | kỳ vọng 4 | đường đi: RRL -> [4, 4]
2 | kỳ vọng 2 | đường đi: RRLRR -> [2, 4, 3]


## 8. TRACE từng bước

Trace nhị phân và hàm kiểm tra trên ví dụ 2.

In [9]:
def trace(points, m):
    def ok(x, verbose=True):
        n, moves, prev = len(points), 0, 0
        parts = []
        for i in range(n):
            need = (x + points[i] - 1) // points[i]
            k = need - prev
            if i == n - 1 and k <= 0:
                parts.append(f"ô {i}: cần {need}, đã ké {prev} -> không cần vào")
                break
            k = max(k, 1)
            moves += 2 * k - 1
            parts.append(f"ô {i}: cần {need}, ké {prev}, vào+nảy k={k} -> +{2 * k - 1} bước (tổng {moves})")
            prev = k - 1
            if moves > m:
                break
        res = moves <= m
        print(f"x = {x}: " + '; '.join(parts) + f" => {'được' if res else 'không'}")
        return res
    lo, hi = 0, min(points) * m
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if ok(mid): lo = mid
        else: hi = mid - 1
    print('đáp án:', lo)

trace([1, 2, 3], 5)

x = 3: ô 0: cần 3, ké 0, vào+nảy k=3 -> +5 bước (tổng 5); ô 1: cần 2, ké 2, vào+nảy k=1 -> +1 bước (tổng 6) => không
x = 1: ô 0: cần 1, ké 0, vào+nảy k=1 -> +1 bước (tổng 1); ô 1: cần 1, ké 0, vào+nảy k=1 -> +1 bước (tổng 2); ô 2: cần 1, ké 0, vào+nảy k=1 -> +1 bước (tổng 3) => được
x = 2: ô 0: cần 2, ké 0, vào+nảy k=2 -> +3 bước (tổng 3); ô 1: cần 1, ké 1, vào+nảy k=1 -> +1 bước (tổng 4); ô 2: cần 1, ké 0, vào+nảy k=1 -> +1 bước (tổng 5) => được
đáp án: 2


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def maxScore(self, points: List[int], m: int) -> int:
        n = len(points)

        def ok(x):
            moves = prev = 0
            for i in range(n):
                k = (x + points[i] - 1) // points[i] - prev
                if i == n - 1 and k <= 0:
                    break
                if k < 1:
                    k = 1
                moves += 2 * k - 1
                prev = k - 1
                if moves > m:
                    return False
            return True

        lo, hi = 0, min(points) * m
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if ok(mid):
                lo = mid
            else:
                hi = mid - 1
        return lo

for p, m, exp in examples:
    got = Solution().maxScore(p, m)
    print(got, 'OK' if got == exp else 'SAI')

4 OK
2 OK


## 10. Vì sao thuật toán đúng

1. **Đơn điệu** ⇒ nhị phân đúng; `0` luôn đạt; `min(points) · m` là cận trên vì ô có `points` nhỏ nhất được ghé ≤ `m` lần.
2. **Cận dưới số bước (lập luận trao đổi):** xét bất kỳ dãy bước hợp lệ. Gọi `c_i` là số lần đi **qua cạnh** `(i, i+1)` (cả hai chiều). Số lần ghé ô `i` (với `0 < i < n−1`) bằng `(c_{i−1} + c_i) / 2` (mỗi lần ghé dùng một cạnh vào, một cạnh ra, trừ điểm cuối). Tham lam chọn `c_i` **nhỏ nhất có thể** từ trái sang: `c_i` chỉ cần đủ để bù phần ô `i` còn thiếu sau khi đã nhận từ cạnh trái. Tăng `c_i` hơn mức đó không bao giờ giảm tổng số bước cần cho các ô bên phải nhiều hơn chi phí đã bỏ ra (mỗi lần qua cạnh thêm cho ô `i+1` tối đa một lần ghé, trong khi tốn một bước — đúng bằng chi phí tự ghé ô `i+1` khi đi tiếp). Vì vậy tham lam đạt tổng bước nhỏ nhất.
3. **Ô cuối:** không bắt buộc phải đặt chân tới nếu đã đủ lần ghé (dãy bước có thể kết thúc ở ô `n − 2`).
4. Kiểm chứng: so với vét cạn mọi dãy bước (mục 12) và dựng đường đi cụ thể (cách 4).

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Thử mọi dãy bước | `O(2^m · n)` | `O(m + n)` |
| 2. Dò `x` tuyến tính | `O(ans · n)` | `O(1)` |
| 3. Nhị phân + tham lam | `O(n · log(min(points) · m))` ≈ `n · 50` | `O(1)` |
| 4. Dựng đường đi | `O(m · log)` | `O(m)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'linear': max_score_linear, 'bs': max_score_bs, 'constructive': max_score_constructive,
         'submit': lambda p, m: Solution().maxScore(p, m)}
edge = [
    ([5, 5], 1, 0),            # không ghé được ô 1
    ([5, 5], 2, 5),
    ([1, 1], 4, 2),            # R L R L? không được: -1 -> 0 -> 1 -> 0 -> 1: [2, 2]
    ([3, 1, 3], 3, 1),
    ([1, 2, 3], 2, 0),         # m < n
]
for p, m, exp in examples + edge:
    assert max_score_brute(p, m) == exp, (p, m, max_score_brute(p, m))
    for name, f in impls.items():
        assert f(p, m) == exp, (name, p, m)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [12]:
import random
rng = random.Random(3449)
cnt = 0
for _ in range(300):
    n = rng.randint(2, 4)
    m = rng.randint(1, 11)
    p = [rng.randint(1, 5) for _ in range(n)]
    exp = max_score_brute(p, m)
    for name, f in impls.items():
        assert f(p, m) == exp, (name, p, m)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ ngẫu nhiên.')

for _ in range(200):          # cỡ vừa: kiểm chứng bằng dựng đường đi
    n = rng.randint(2, 30); m = rng.randint(1, 300)
    p = [rng.randint(1, 20) for _ in range(n)]
    assert max_score_bs(p, m) == max_score_constructive(p, m)
print('Nhị phân + tham lam khớp dựng đường đi trên 200 bộ cỡ vừa.')

Khớp vét cạn trên 300 bộ ngẫu nhiên.
Nhị phân + tham lam khớp dựng đường đi trên 200 bộ cỡ vừa.


### 12.3. Ca lớn `n = 5·10^4`, `m = 10^9`

In [13]:
import time
n = 5 * 10 ** 4
cases = {
    'ngẫu nhiên': [rng.randint(1, 10 ** 6) for _ in range(n)],
    'toàn 1': [1] * n,
    'toàn 10^6': [10 ** 6] * n,
    'xen kẽ 1 / 10^6': [1 if i % 2 else 10 ** 6 for i in range(n)],
}
for name, p in cases.items():
    for m in (3 * n, 10 ** 9):
        t = time.time(); a = Solution().maxScore(p, m); t1 = time.time() - t
        print(f"{name:16s} m={m:>10d}: {a:>16d} | {t1:.2f}s")

ngẫu nhiên       m=    150000:           102018 | 0.11s


ngẫu nhiên       m=1000000000:        879403215 | 0.23s
toàn 1           m=    150000:                3 | 0.02s
toàn 1           m=1000000000:            20000 | 0.10s


toàn 10^6        m=    150000:          3000000 | 0.11s


toàn 10^6        m=1000000000:      20000000000 | 0.29s
xen kẽ 1 / 10^6  m=    150000:                3 | 0.02s
xen kẽ 1 / 10^6  m=1000000000:            20000 | 0.11s


## 13. Các lỗi người mới hay gặp

Hàm tìm phản ví dụ (seed cố định, so với vét cạn):

In [14]:
def find_counterexample(wrong, seed=0, tries=3000):
    rng = random.Random(seed)
    for _ in range(tries):
        n = rng.randint(2, 4); m = rng.randint(1, 10)
        p = [rng.randint(1, 4) for _ in range(n)]
        if wrong(p, m) != max_score_brute(p, m):
            return p, m
    return None

def make_solver(last_special=True, force_step=True, credit_next=True, hi_fn=lambda p, m: min(p) * m):
    def solve(points, m):
        n = len(points)
        def ok(x):
            moves = prev = 0
            for i in range(n):
                k = (x + points[i] - 1) // points[i] - prev
                if last_special and i == n - 1 and k <= 0:
                    break
                if force_step:
                    k = max(k, 1)
                elif k <= 0:
                    prev = 0
                    continue
                moves += 2 * k - 1
                prev = k - 1 if credit_next else 0
                if moves > m:
                    return False
            return True
        return max_true(0, hi_fn(points, m), ok)
    return solve

### Lỗi 1 — Luôn bước vào ô cuối, kể cả khi nó đã đủ lần ghé

In [15]:
wrong_last = make_solver(last_special=False)
ce = find_counterexample(wrong_last, seed=1)
print('phản ví dụ:', ce, '| sai:', wrong_last(*ce), '| đúng:', max_score_brute(*ce))

phản ví dụ: ([1, 4], 9) | sai: 4 | đúng: 5


**Vì sao sai:** dãy bước có thể kết thúc ở ô `n − 2`; nếu các lần nảy đã đủ cho ô `n − 1`, bước thêm vào ô cuối là lãng phí.

### Lỗi 2 — Bỏ qua ô `i` (không tính bước) khi nó đã đủ lần ghé

In [16]:
wrong_skip = make_solver(force_step=False)
ce = find_counterexample(wrong_skip, seed=2)
print('phản ví dụ:', ce, '| sai:', wrong_skip(*ce), '| đúng:', max_score_brute(*ce))

phản ví dụ: ([1, 2, 2], 4) | sai: 2 | đúng: 1


**Vì sao sai:** muốn tới các ô bên phải vẫn phải **đi qua** ô `i` — tốn ít nhất 1 bước.

### Lỗi 3 — Không tính các lần ghé "ké" cho ô `i + 1`

In [17]:
wrong_credit = make_solver(credit_next=False)
ce = find_counterexample(wrong_credit, seed=3)
print('phản ví dụ:', ce, '| sai:', wrong_credit(*ce), '| đúng:', max_score_brute(*ce))

phản ví dụ: ([2, 3], 10) | sai: 6 | đúng: 10


**Vì sao sai:** mỗi lần nảy `i → i+1 → i` đều đặt chân lên ô `i + 1` và cộng điểm cho nó.

### Lỗi 4 — Cận trên nhị phân quá nhỏ (`max(points)`)

In [18]:
wrong_hi = make_solver(hi_fn=lambda p, m: max(p))
print('sai :', wrong_hi([1, 1], 100), '| đúng:', Solution().maxScore([1, 1], 100))

sai : 1 | đúng: 50


**Vì sao sai:** mỗi ô có thể được ghé nhiều lần, nên đáp án có thể vượt xa `max(points)` (ở đây 50).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tối đa hóa giá trị nhỏ nhất" ⇒ nhị phân đáp án.
- Di chuyển trên đường thẳng, ô cần ghé nhiều lần ⇒ **nảy qua lại** giữa hai ô kề nhau, xử lý từ trái sang.
- Lần nảy tạo "lợi ích phụ" cho ô kế ⇒ mang biến `prev` sang bước sau.
- Cẩn thận ô biên (ô cuối không bắt buộc đặt chân).

## 15. Biến thể của bài

1. Bắt đầu ở giữa mảng ⇒ phải xử lý hai phía.
2. Mỗi bước tốn chi phí khác nhau ⇒ hàm kiểm tra tính tổng chi phí thay vì số bước.
3. Tối đa hóa **tổng** điểm thay vì min ⇒ đi tới ô lớn nhất kề nhau rồi nảy giữa hai ô có tổng lớn nhất.
4. Bài liên quan: LC 1552, LC 2528 (Maximize the Minimum Powered City) — cùng khuôn nhị phân + tham lam.

In [19]:
# Biến thể 3: tối đa hóa TỔNG điểm với đúng tối đa m bước (bắt đầu từ -1)
def max_total(points, m):
    n = len(points)
    best, pref = 0, 0
    for j in range(min(n, m)):            # đi thẳng tới ô j (j+1 bước), rồi nảy giữa j và j-1 hoặc j+1
        pref += points[j]
        rest = m - (j + 1)
        cand = pref
        for q in (j - 1, j + 1):
            if 0 <= q < n:
                pair = points[j] + points[q]
                extra = (rest // 2) * pair + (points[q] if rest % 2 else 0)   # nảy j <-> q
                cand = max(cand, pref + extra)
        best = max(best, cand)
    return best

def max_total_brute(points, m):
    n = len(points); best = 0
    def dfs(pos, used, s):
        nonlocal best
        best = max(best, s)
        if used == m:
            return
        for q in (pos - 1, pos + 1):
            if 0 <= q < n:
                dfs(q, used + 1, s + points[q])
    dfs(-1, 0, 0)
    return best

rng = random.Random(15)
for _ in range(300):
    p = [rng.randint(1, 6) for _ in range(rng.randint(2, 5))]
    m = rng.randint(1, 10)
    assert max_total(p, m) == max_total_brute(p, m), (p, m)
print('Biến thể tổng điểm khớp vét cạn:', max_total([1, 5, 2, 9], 7))

Biến thể tổng điểm khớp vét cạn: 30


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Thử mọi dãy bước | `O(2^m n)` | Oracle |
| 2 | Tham lam + dò tuyến tính | `O(ans · n)` | Chỉ khi đáp án nhỏ |
| 3 | Nhị phân + tham lam | `O(n log)` | **Nộp bài** |
| 4 | Dựng đường đi + mô phỏng | `O(m log)` | Kiểm chứng, ra lời giải cụ thể |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `points = [1, 1, 1]`, `m = 3`.

**Bài 2 (tính tay):** `points = [2, 1]`, `m = 5`.

**Bài 3 (điền TODO):** viết `min_moves(points, x)` — số bước ít nhất để mọi ô `≥ x`.

**Bài 4 (tự viết lại):** biến thể 1 — bắt đầu ở chỉ số `n` (bên phải), bước đầu tiên vào ô `n − 1`.

**Bài 5 (mini challenge):** với mỗi `m` từ 1 tới `M`, in đáp án — dùng `min_moves` và nhị phân ngược (`x` lớn nhất có `min_moves ≤ m`).

In [20]:
def min_moves_todo(points, x):
    # TODO
    raise NotImplementedError

def max_score_from_right_todo(points, m):
    # TODO
    raise NotImplementedError

def answers_for_all_m_todo(points, M):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Ba ô, ba bước. (2) Đi thẳng `R R R`. (3) Mỗi ô 1 điểm ⇒ 1.

**Bài 2.** (1) Thử `x = 2`: `need = [1, 2]`. (2) Ô 0: 1 bước; ô 1: 1 bước + 1 lần nảy (2 bước). (3) Thử `x = 3`.

**Bài 3.** (1) Chính là biến `moves` trong hàm kiểm tra, không dừng sớm. (2) `x = 0` ⇒ 0 bước. (3) So sánh với vét cạn số bước nhỏ nhất.

**Bài 4.** (1) Đảo ngược mảng. (2) Gọi lại lời giải. (3) Kiểm tra bằng vét cạn bắt đầu từ `n`.

**Bài 5.** (1) `min_moves(x)` không giảm theo `x`. (2) Với mỗi `m`, nhị phân `x`. (3) Hoặc quét `x` tăng dần và hai con trỏ trên `m`.

## 19. Đáp án tham khảo cho bài tập

In [21]:
# Bài 1, 2, 3
a1 = Solution().maxScore([1, 1, 1], 3)
a2 = Solution().maxScore([2, 1], 5)
assert a1 == max_score_brute([1, 1, 1], 3) == 1 and a2 == max_score_brute([2, 1], 5)
print('Bài 1:', a1, '| Bài 2:', a2)

def min_moves(points, x):
    n, moves, prev = len(points), 0, 0
    if x <= 0:
        return 0
    for i in range(n):
        k = (x + points[i] - 1) // points[i] - prev
        if i == n - 1 and k <= 0:
            break
        k = max(k, 1)
        moves += 2 * k - 1
        prev = k - 1
    return moves

def min_moves_brute(points, x, cap=14):
    from collections import deque
    n = len(points)
    start = (-1, tuple([0] * n))
    dq, seen = deque([(start, 0)]), {start}
    while dq:
        (pos, sc), d = dq.popleft()
        if min(sc) >= x:
            return d
        if d == cap:
            continue
        for q in (pos - 1, pos + 1):
            if 0 <= q < n:
                s = list(sc); s[q] = min(x, s[q] + points[q])
                st = (q, tuple(s))
                if st not in seen:
                    seen.add(st); dq.append((st, d + 1))
    return None

rng = random.Random(3)
for _ in range(200):
    p = [rng.randint(1, 4) for _ in range(rng.randint(2, 4))]
    x = rng.randint(0, 6)
    mb = min_moves_brute(p, x)
    if mb is not None:
        assert min_moves(p, x) == mb, (p, x)
print('Bài 3 khớp BFS số bước nhỏ nhất.')

Bài 1: 1 | Bài 2: 2
Bài 3 khớp BFS số bước nhỏ nhất.


In [22]:
# Bài 4, 5
def max_score_from_right(points, m):
    return Solution().maxScore(points[::-1], m)

def brute_from_right(points, m):
    return max_score_brute(points[::-1], m)

rng = random.Random(4)
for _ in range(150):
    p = [rng.randint(1, 5) for _ in range(rng.randint(2, 4))]; m = rng.randint(1, 10)
    assert max_score_from_right(p, m) == brute_from_right(p, m)
print('Bài 4 khớp vét cạn.')

def answers_for_all_m(points, M):
    res = []
    for m in range(1, M + 1):
        lo, hi = 0, min(points) * m
        while lo < hi:
            mid = (lo + hi + 1) // 2
            if min_moves(points, mid) <= m: lo = mid
            else: hi = mid - 1
        res.append(lo)
    return res

r = answers_for_all_m([1, 2, 3], 12)
assert r == [Solution().maxScore([1, 2, 3], m) for m in range(1, 13)]
print('Bài 5: đáp án theo m = 1..12 cho [1,2,3]:', r)

Bài 4 khớp vét cạn.
Bài 5: đáp án theo m = 1..12 cho [1,2,3]: [0, 0, 1, 1, 2, 2, 3, 3, 3, 3, 4, 4]


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
def ok(x):
    moves = prev = 0
    for i in range(n):
        k = ceil(x / p[i]) - prev
        if i == n-1 and k <= 0: break
        k = max(k, 1)
        moves += 2*k - 1; prev = k - 1
        if moves > m: return False
    return True
nhị phân x trong [0, min(p) * m]
```

**Checklist:**

- [ ] Ô cuối: đủ rồi thì không bước vào?
- [ ] Ô giữa: luôn tốn ít nhất 1 bước để đi qua?
- [ ] Lần nảy ghé ké ô `i + 1` (`prev = k − 1`)?
- [ ] Cận trên `min(points) · m`, dùng `⌈x/p⌉` bằng số nguyên?
- [ ] Dừng sớm khi `moves > m` để không chậm?